In [15]:
from scipy.io import loadmat
import numpy as np

from pathlib import Path
import re
import shutil
from tqdm import tqdm
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from torchvision.io import decode_image
from torch.utils.data import Dataset
from torchinfo import summary
import os

## Data splits

In [16]:
def ensure_split_folder(split_dir, split_ids, labels, source_dir="./data/jpg"):
    source_path = Path(source_dir)
    target_path = Path(split_dir)

    if target_path.exists():
        print(f"{target_path} already exists; skipping creation.")
        return target_path

    target_path.mkdir(parents=True, exist_ok=True)
    split_id_set = set(np.asarray(split_ids).reshape(-1).astype(int).tolist())
    labels_arr = np.asarray(labels).reshape(-1).astype(int)

    copied = 0
    for file_path in tqdm(source_path.iterdir()):
        if not file_path.is_file():
            continue

        match = re.search(r"(\d+)(?!.*\d)", file_path.stem)
        if not match:
            continue

        sample_id = int(match.group(1))
        if sample_id not in split_id_set:
            continue

        label = int(labels_arr[sample_id - 1])
        label_dir = target_path / str(label)
        label_dir.mkdir(parents=True, exist_ok=True)

        shutil.copy2(file_path, label_dir / file_path.name)
        copied += 1

    print(f"Created {target_path} with {copied} files.")
    return target_path


In [17]:
# define directory names
root_dir = './data'
src_dir = './data/jpg' # this is just where i have all the pics
train_dir = './data/train'
val_dir = './data/val'
test_dir = './data/test'

# load in set splits
setid = loadmat("./data/setid.mat")
labels = loadmat("./data/imagelabels.mat")['labels']
train = setid['trnid']
val = setid['valid']
test = setid['tstid']

# split data into train, val, test
# we just copyin for now since the dataset is so small
ensure_split_folder(train_dir, train, labels)
ensure_split_folder(val_dir, val, labels)
ensure_split_folder(test_dir, test, labels)


data/train already exists; skipping creation.
data/val already exists; skipping creation.
data/test already exists; skipping creation.


PosixPath('data/test')

## DataLoader definitions

In [4]:
# first calculate stats on the train set for transformation normalization
mean = 0
std = 0
n = 0

train_dir = Path("./data/train")
all_paths = [p for p in train_dir.rglob("*") if p.is_file()]

for path in all_paths:
    img = decode_image(str(path)).float()

    img = img.view(3, -1)
    mean += img.mean(1)
    std += img.std(1)
    n += 1

mean /= n
std /= n


In [18]:
# two views for contrastive objective
class TwoViewTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        view1 = self.base_transform(x)
        view2 = self.base_transform(x)
        return view1, view2
    
simclr_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomApply(
        [T.ColorJitter(0.4, 0.4, 0.4, 0.1)],
        p=0.8
    ),
    #T.RandomGrayscale(p=0.2), # OG SimCLR doesnt use this
    T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),
    T.ToTensor(),
    T.Normalize(mean, std)
])

train_dataset = ImageFolder(
    root=train_dir,
    transform=TwoViewTransform(simclr_transform)
)

val_dataset = ImageFolder(
    root=val_dir,
    transform=TwoViewTransform(simclr_transform)
)

test_dataset = ImageFolder(
    root=test_dir,
    transform=TwoViewTransform(simclr_transform)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

## SimCLR Model

In [19]:
class SimCLR(nn.Module):
    def __init__(self, out_features=128):
        super().__init__()
        self.resnet18 = resnet18()
        
        # replace last layer with linear projection head
        num_features = self.resnet18.fc.in_features
        self.resnet18.fc = nn.Linear(num_features, num_features)

        self.relu = nn.ReLU()
        self.fc = nn.Linear(num_features, out_features)        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y_pred = self.resnet18(x)
        y_pred = self.relu(y_pred)
        y_pred = self.fc(y_pred)
        return y_pred
    
    def representation(self, x: torch.Tensor) -> torch.Tensor:
        return self.resnet18(x)

In [ ]:
def nt_xent(z1: torch.Tensor, z2: torch.Tensor):
    

## Train SimCLR

In [14]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
simCLR = SimCLR().to(device)
lr = 1e-3
epochs = 10
opt = torch.optim.Adam(simCLR.parameters(), lr=lr)

simCLR.train()
for ep in range(1, epochs + 1):
    for (x1, x2), _ in train_loader:

        x1 = x1.to(device)
        x2 = x2.to(device)

        z1 = simCLR(x1)
        z2 = simCLR(x2)

        loss = nt_xent(z1, z2)

        opt.zero_grad()
        loss.backward()
        opt.step()

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

RuntimeError: DataLoader worker (pid(s) 79667, 79670) exited unexpectedly